# IS 445 HW5: Bigfoot sighting reports with Altair

Raymond Lu (GitHub: rl38-lab)

This notebook builds the two plots shown on my Jekyll page
(https://rl38-lab.github.io/projects/2-bigfoot-sightings). The data are the BFRO (Bigfoot Field Researchers
Organization) sighting reports from the course data repository, read directly from a URL.

In [1]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()  # just in case; the plotted tables are small anyway

## 1. Read the data from a URL

In [2]:
data_url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv"
bf = pd.read_csv(data_url)
print(bf.shape)
bf[['date', 'season', 'state', 'classification']].head()

(4747, 29)


## 2. Data transformations

- Parse the `date` strings into timestamps and pull out the year. About 950 reports have no usable date; those are dropped from the time-based plot only.
- Keep reports from 1950 to 2018. Before 1950 there are only a handful of reports per decade, which would stretch the x-axis without adding information.
- Count reports per (year, season, classification). The plot then works from this small summary table (a few hundred rows) instead of all 4,747 raw reports, which keeps the exported JSON small.
- For the second plot, keep the 20 states with the most reports and drop the 91 reports whose season is "Unknown".

In [3]:
bf['date'] = pd.to_datetime(bf['date'], errors='coerce')
bf['year'] = bf['date'].dt.year

print('reports with no usable date:', bf['year'].isna().sum())
print(bf['classification'].value_counts())
print(bf['season'].value_counts())

reports with no usable date: 950
classification
Class B    2370
Class A    2347
Class C      30
Name: count, dtype: int64
season
Summer     1765
Fall       1413
Spring      767
Winter      711
Unknown      91
Name: count, dtype: int64


In [4]:
season_order = ['Spring', 'Summer', 'Fall', 'Winter']

dated = bf.dropna(subset=['year']).copy()
dated['year'] = dated['year'].astype(int)
dated = dated[(dated['year'] >= 1950) & (dated['year'] <= 2018)]
dated = dated[dated['season'].isin(season_order)]

yearly = (dated.groupby(['year', 'season', 'classification'])
               .size()
               .reset_index(name='reports'))
print(yearly.shape)
yearly.head()

(451, 4)


In [5]:
# colors for the three report classes (categorical data -> distinct hues)
class_domain = ['Class A', 'Class B', 'Class C']
class_range = ['#d95f02', '#1b9e77', '#7570b3']
class_scale = alt.Scale(domain=class_domain, range=class_range)

## 3. Plot 1: reports over time, linked to a season breakdown

Interactivity:
- Drag across the top chart to select a range of years (an interval brush on the x-axis). The bottom bar chart then only counts reports from those years.
- Click a class in the legend to highlight it in both charts (shift-click to add more classes; double-click empty space to reset).

In [6]:
brush = alt.selection_interval(encodings=['x'], name='years')
legend_pick = alt.selection_point(fields=['classification'], bind='legend', name='cls')

base = alt.Chart(yearly)

timeline = base.mark_line(point=alt.OverlayMarkDef(size=18)).encode(
    x=alt.X('year:Q', title='Year of sighting', axis=alt.Axis(format='d'),
            scale=alt.Scale(domain=[1950, 2018], nice=False)),
    y=alt.Y('sum(reports):Q', title='Reports per year'),
    color=alt.Color('classification:N', scale=class_scale, title='Report class'),
    opacity=alt.condition(legend_pick, alt.value(1), alt.value(0.15)),
    tooltip=[alt.Tooltip('year:Q', title='Year'),
             alt.Tooltip('classification:N', title='Class'),
             alt.Tooltip('sum(reports):Q', title='Reports')]
).add_params(brush, legend_pick).properties(
    width=620, height=230,
    title='Bigfoot reports per year by report class (drag to select years)'
)

season_bars = base.mark_bar().encode(
    x=alt.X('season:N', sort=season_order, title='Season of sighting', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('sum(reports):Q', title='Reports in selected years'),
    color=alt.Color('classification:N', scale=class_scale, title='Report class'),
    order=alt.Order('classification:N', sort='ascending'),
    opacity=alt.condition(legend_pick, alt.value(1), alt.value(0.15)),
    tooltip=[alt.Tooltip('season:N', title='Season'),
             alt.Tooltip('classification:N', title='Class'),
             alt.Tooltip('sum(reports):Q', title='Reports')]
).transform_filter(brush).properties(
    width=620, height=200,
    title='Reports by season for the selected years'
)

plot1 = alt.vconcat(timeline, season_bars).configure_axis(grid=True, gridOpacity=0.4)
plot1

In [7]:
# saved into the Jekyll site's assets folder (this notebook lives in python_notebooks/)
plot1.save('../assets/json/bigfoot_timeline_seasons.json')

## 4. Plot 2: which states report the most sightings, and in which season

Interactivity: a dropdown menu filters the heatmap to all reports, Class A only, Class B only, or Class C only.

In [8]:
seasonal = bf[bf['season'].isin(season_order)]
top_states = seasonal['state'].value_counts().head(20).index

state_season = (seasonal[seasonal['state'].isin(top_states)]
                .groupby(['state', 'season', 'classification'])
                .size()
                .reset_index(name='reports'))
print(state_season.shape)
state_season.head()

(173, 4)


In [9]:
class_dropdown = alt.binding_select(options=['All', 'Class A', 'Class B', 'Class C'],
                                    name='Report class: ')
class_param = alt.param(name='pick_class', value='All', bind=class_dropdown)

heat_base = alt.Chart(state_season).add_params(class_param).transform_filter(
    "pick_class == 'All' || datum.classification == pick_class"
).transform_aggregate(
    reports='sum(reports)', groupby=['state', 'season']
)

heat = heat_base.mark_rect(stroke='white', strokeWidth=1).encode(
    x=alt.X('season:O', sort=season_order, title='Season of sighting',
            axis=alt.Axis(labelAngle=0, orient='top')),
    y=alt.Y('state:N', sort=alt.EncodingSortField(field='reports', op='sum', order='descending'),
            title=None),
    color=alt.Color('reports:Q', title='Reports',
                    scale=alt.Scale(scheme='yelloworangebrown')),
    tooltip=[alt.Tooltip('state:N', title='State'),
             alt.Tooltip('season:O', title='Season'),
             alt.Tooltip('reports:Q', title='Reports')]
)

labels = heat_base.mark_text(fontSize=11).encode(
    x=alt.X('season:O', sort=season_order),
    y=alt.Y('state:N', sort=alt.EncodingSortField(field='reports', op='sum', order='descending')),
    text='reports:Q',
    color=alt.condition('datum.reports > 120', alt.value('white'), alt.value('black'))
)

plot2 = (heat + labels).properties(
    width=360, height=520,
    title='Bigfoot reports by state and season (top 20 states)'
)
plot2

In [10]:
plot2.save('../assets/json/bigfoot_state_season_heatmap.json')